# 双均线策略量化流程

完整流程：
1. 数据加载与筛选（市值100-500亿主板股票）
2. 时间序列数据集划分（训练/验证/测试，无穿越）
3. 双均线策略回测
4. 结果分析

In [ ]:
# 环境初始化
%run jupyter_setup.py

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("✅ 环境准备完成")

## 第一步：加载A股历史数据

In [ ]:
# 加载数据
DATA_PATH = './data/a_stock_history_price_20260223.csv'

print("📊 加载A股历史数据...")
df = pd.read_csv(DATA_PATH)

# 数据预处理
df['date'] = pd.to_datetime(df['date'])
df['code'] = df['code'].astype(str).str.zfill(6)

# 确保市值字段存在（如果没有，尝试其他字段）
if 'market_cap' not in df.columns:
    # 尝试从其他字段计算或查找
    print("⚠️ 市值字段不存在，尝试从现有字段获取...")
    # 如果有收盘价和总股本，可以计算市值
    pass

print(f"✅ 数据加载完成")
print(f"   总记录数: {len(df):,}")
print(f"   股票数量: {df['code'].nunique()}")
print(f"   日期范围: {df['date'].min().date()} ~ {df['date'].max().date()}")
print(f"\n数据列: {list(df.columns)}")

In [ ]:
# 查看数据样本
df.head()

## 第二步：筛选股票（市值100-500亿，主板）

In [ ]:
def filter_stocks(df, min_cap=100e8, max_cap=500e8, n_samples=50, random_seed=42):
    """
    筛选符合条件的股票
    
    Parameters:
    -----------
    df : DataFrame
    min_cap : float
        最小市值（默认100亿）
    max_cap : float
        最大市值（默认500亿）
    n_samples : int
        随机选择数量
    random_seed : int
        随机种子
    """
    
    print("\n🔍 开始筛选股票...")
    
    # 1. 获取最新日期的数据（用于筛选市值）
    latest_date = df['date'].max()
    latest_df = df[df['date'] == latest_date].copy()
    
    print(f"   最新日期: {latest_date.date()}")
    print(f"   当日股票数: {len(latest_df)}")
    
    # 2. 判断主板股票（代码规则）
    # 上海主板: 600/601/603/605开头
    # 深圳主板: 000/001/002开头
    def is_main_board(code):
        code = str(code).zfill(6)
        return (code.startswith('6') or  # 上海主板
                code.startswith('000') or  # 深圳主板
                code.startswith('001') or
                code.startswith('002'))
    
    latest_df['is_main_board'] = latest_df['code'].apply(is_main_board)
    main_board = latest_df[latest_df['is_main_board']].copy()
    
    print(f"   主板股票数: {len(main_board)}")
    
    # 3. 筛选市值
    # 注意：根据实际数据字段调整
    if 'market_cap' in main_board.columns:
        # 使用市值字段（假设单位为元）
        filtered = main_board[(main_board['market_cap'] >= min_cap) & 
                             (main_board['market_cap'] <= max_cap)]
    else:
        # 如果没有市值字段，使用收盘价作为代理（实际应用中应该用总股本*股价）
        print("   ⚠️ 市值字段不存在，使用收盘价作为筛选参考")
        # 假设股价10-50元对应合理市值范围（仅为示例）
        filtered = main_board[(main_board['close'] >= 10) & 
                             (main_board['close'] <= 50)]
    
    print(f"   市值筛选后: {len(filtered)}")
    
    # 4. 随机选择
    if len(filtered) > n_samples:
        np.random.seed(random_seed)
        selected = filtered.sample(n=n_samples)
    else:
        selected = filtered
    
    selected_codes = selected['code'].unique().tolist()
    
    print(f"\n✅ 最终选中 {len(selected_codes)} 只股票")
    print(f"   代码示例: {', '.join(selected_codes[:10])}...")
    
    return selected_codes, selected

# 执行筛选
selected_codes, selected_info = filter_stocks(df, min_cap=100e8, max_cap=500e8, n_samples=50)

# 显示选中的股票
print("\n📋 选中股票列表:")
display_cols = ['code', 'name', 'close']
if 'market_cap' in selected_info.columns:
    selected_info['market_cap_yi'] = selected_info['market_cap'] / 1e8
    display_cols.append('market_cap_yi')
print(selected_info[display_cols].to_string(index=False))

In [ ]:
# 过滤完整数据集，只保留选中的股票
print("\n🔍 过滤完整数据集...")
filtered_df = df[df['code'].isin(selected_codes)].copy()

print(f"   原始数据: {len(df):,} 条")
print(f"   过滤后: {len(filtered_df):,} 条")
print(f"   股票数: {filtered_df['code'].nunique()}")

# 按股票代码和日期排序
filtered_df = filtered_df.sort_values(['code', 'date'])
filtered_df.head()

## 第三步：时间序列数据集划分（严格无穿越）

In [ ]:
def time_series_split(df, train_ratio=0.6, val_ratio=0.2, test_ratio=0.2):
    """
    按时间序列划分数据集（严格无穿越）
    
    划分方式:
    - 训练集: 最早 60% 的时间
    - 验证集: 中间 20% 的时间
    - 测试集: 最后 20% 的时间
    
    Parameters:
    -----------
    df : DataFrame
        包含所有股票的数据
    train/val/test_ratio : float
        划分比例
    """
    
    print("\n⏰ 时间序列数据集划分...")
    
    # 获取全局日期范围
    all_dates = sorted(df['date'].unique())
    n_dates = len(all_dates)
    
    print(f"   总交易日数: {n_dates}")
    print(f"   日期范围: {all_dates[0]} ~ {all_dates[-1]}")
    
    # 计算划分点
    train_end_idx = int(n_dates * train_ratio)
    val_end_idx = int(n_dates * (train_ratio + val_ratio))
    
    train_end_date = all_dates[train_end_idx]
    val_end_date = all_dates[val_end_idx]
    
    # 划分数据集
    train_df = df[df['date'] <= train_end_date].copy()
    val_df = df[(df['date'] > train_end_date) & (df['date'] <= val_end_date)].copy()
    test_df = df[df['date'] > val_end_date].copy()
    
    print(f"\n📊 划分结果:")
    print(f"   训练集: {train_df['date'].min().date()} ~ {train_df['date'].max().date()} ({len(train_df):,} 条)")
    print(f"   验证集: {val_df['date'].min().date()} ~ {val_df['date'].max().date()} ({len(val_df):,} 条)")
    print(f"   测试集: {test_df['date'].min().date()} ~ {test_df['date'].max().date()} ({len(test_df):,} 条)")
    
    # 验证无穿越
    assert train_df['date'].max() < val_df['date'].min(), "训练集和验证集有重叠！"
    assert val_df['date'].max() < test_df['date'].min(), "验证集和测试集有重叠！"
    
    print(f"\n✅ 验证通过：无时间穿越")
    
    return train_df, val_df, test_df

# 执行划分
train_df, val_df, test_df = time_series_split(filtered_df)

# 保存划分后的数据
train_df.to_csv('./data/train_data.csv', index=False)
val_df.to_csv('./data/val_data.csv', index=False)
test_df.to_csv('./data/test_data.csv', index=False)
print("\n💾 数据已保存到 data/ 目录")

In [ ]:
# 可视化数据划分
fig, ax = plt.subplots(figsize=(12, 6))

# 统计每天的股票数量
train_daily = train_df.groupby('date').size()
val_daily = val_df.groupby('date').size()
test_daily = test_df.groupby('date').size()

ax.fill_between(train_daily.index, 0, train_daily.values, alpha=0.5, label='Train', color='blue')
ax.fill_between(val_daily.index, 0, val_daily.values, alpha=0.5, label='Validation', color='orange')
ax.fill_between(test_daily.index, 0, test_daily.values, alpha=0.5, label='Test', color='green')

ax.axvline(train_df['date'].max(), color='red', linestyle='--', alpha=0.7, label='Train/Val Split')
ax.axvline(val_df['date'].max(), color='red', linestyle='--', alpha=0.7, label='Val/Test Split')

ax.set_xlabel('Date')
ax.set_ylabel('Number of Records')
ax.set_title('Time Series Data Split (No Look-ahead)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('./data/time_series_split.png', dpi=150)
plt.show()

print("✅ 可视化已保存")

## 第四步：双均线策略回测

In [ ]:
class DualMAStrategy:
    """
    双均线策略
    
    买入信号: 短期均线上穿长期均线（金叉）
    卖出信号: 短期均线下穿长期均线（死叉）
    """
    
    def __init__(self, short_window=5, long_window=20):
        self.short_window = short_window
        self.long_window = long_window
    
    def generate_signals(self, df):
        """生成交易信号"""
        df = df.copy()
        
        # 计算均线
        df['ma_short'] = df['close'].rolling(window=self.short_window).mean()
        df['ma_long'] = df['close'].rolling(window=self.long_window).mean()
        
        # 生成信号
        df['signal'] = 0
        df.loc[df['ma_short'] > df['ma_long'], 'signal'] = 1  # 多头
        df.loc[df['ma_short'] <= df['ma_long'], 'signal'] = -1  # 空头或观望
        
        # 交易信号（信号变化时触发）
        df['position'] = df['signal'].diff()
        df.loc[df['position'] > 0, 'action'] = 'buy'
        df.loc[df['position'] < 0, 'action'] = 'sell'
        
        return df
    
    def backtest(self, df, init_capital=100000, commission=0.001):
        """
        回测策略
        
        Returns:
        --------
        result : dict
            回测结果
        """
        df = self.generate_signals(df)
        
        # 初始化
        capital = init_capital
        position = 0
        trades = []
        
        for idx, row in df.iterrows():
            if row['action'] == 'buy' and capital > 0:
                # 买入
                shares = capital * (1 - commission) / row['close']
                cost = shares * row['close']
                trades.append({
                    'date': row['date'],
                    'action': 'buy',
                    'price': row['close'],
                    'shares': shares,
                    'cost': cost
                })
                position = shares
                capital = 0
                
            elif row['action'] == 'sell' and position > 0:
                # 卖出
                revenue = position * row['close'] * (1 - commission)
                trades.append({
                    'date': row['date'],
                    'action': 'sell',
                    'price': row['close'],
                    'shares': position,
                    'revenue': revenue
                })
                capital = revenue
                position = 0
        
        # 计算最终市值
        final_value = capital + position * df['close'].iloc[-1]
        total_return = (final_value - init_capital) / init_capital
        
        return {
            'init_capital': init_capital,
            'final_value': final_value,
            'total_return': total_return,
            'trades': pd.DataFrame(trades),
            'n_trades': len([t for t in trades if t['action'] == 'buy']),
            'df': df
        }

print("✅ 双均线策略类定义完成")

In [ ]:
# 对每只股票进行回测
def backtest_all_stocks(data_df, name="数据集"):
    """对所有股票进行回测"""
    
    print(f"\n🚀 回测{name}...")
    
    strategy = DualMAStrategy(short_window=5, long_window=20)
    results = []
    
    for code in data_df['code'].unique():
        stock_df = data_df[data_df['code'] == code].copy()
        stock_df = stock_df.sort_values('date')
        
        if len(stock_df) < 30:  # 数据太少跳过
            continue
        
        try:
            result = strategy.backtest(stock_df)
            results.append({
                'code': code,
                'name': stock_df['name'].iloc[0] if 'name' in stock_df.columns else code,
                'return': result['total_return'],
                'n_trades': result['n_trades'],
                'final_value': result['final_value'],
            })
        except Exception as e:
            print(f"   ⚠️ {code} 回测失败: {e}")
    
    results_df = pd.DataFrame(results)
    
    # 统计
    print(f"\n📊 {name}回测结果:")
    print(f"   回测股票数: {len(results_df)}")
    print(f"   平均收益率: {results_df['return'].mean()*100:.2f}%")
    print(f"   胜率: {(results_df['return'] > 0).mean()*100:.1f}%")
    print(f"   最好股票: {results_df.loc[results_df['return'].idxmax(), 'name']} ({results_df['return'].max()*100:.2f}%)")
    print(f"   最差股票: {results_df.loc[results_df['return'].idxmin(), 'name']} ({results_df['return'].min()*100:.2f}%)")
    
    return results_df

# 分别对三个数据集进行回测
train_results = backtest_all_stocks(train_df, "训练集")
val_results = backtest_all_stocks(val_df, "验证集")
test_results = backtest_all_stocks(test_df, "测试集")

In [ ]:
# 可视化回测结果
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 收益率分布对比
ax = axes[0, 0]
ax.hist(train_results['return']*100, bins=20, alpha=0.5, label='Train', color='blue')
ax.hist(val_results['return']*100, bins=20, alpha=0.5, label='Val', color='orange')
ax.hist(test_results['return']*100, bins=20, alpha=0.5, label='Test', color='green')
ax.axvline(0, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Return (%)')
ax.set_ylabel('Frequency')
ax.set_title('Return Distribution by Dataset')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. 平均收益率对比
ax = axes[0, 1]
datasets = ['Train', 'Val', 'Test']
mean_returns = [
    train_results['return'].mean()*100,
    val_results['return'].mean()*100,
    test_results['return'].mean()*100
]
colors = ['blue', 'orange', 'green']
bars = ax.bar(datasets, mean_returns, color=colors, alpha=0.6)
ax.axhline(0, color='red', linestyle='--', alpha=0.5)
ax.set_ylabel('Average Return (%)')
ax.set_title('Average Return by Dataset')
ax.grid(True, alpha=0.3, axis='y')

# 在柱子上添加数值
for bar, val in zip(bars, mean_returns):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.2f}%', ha='center', va='bottom')

# 3. 胜率对比
ax = axes[1, 0]
win_rates = [
    (train_results['return'] > 0).mean()*100,
    (val_results['return'] > 0).mean()*100,
    (test_results['return'] > 0).mean()*100
]
bars = ax.bar(datasets, win_rates, color=colors, alpha=0.6)
ax.axhline(50, color='red', linestyle='--', alpha=0.5)
ax.set_ylabel('Win Rate (%)')
ax.set_title('Win Rate by Dataset')
ax.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, win_rates):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.1f}%', ha='center', va='bottom')

# 4. 交易次数分布
ax = axes[1, 1]
ax.boxplot([
    train_results['n_trades'],
    val_results['n_trades'],
    test_results['n_trades']
], labels=['Train', 'Val', 'Test'])
ax.set_ylabel('Number of Trades')
ax.set_title('Trade Count Distribution')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('./data/backtest_results_comparison.png', dpi=150)
plt.show()

print("\n✅ 回测结果可视化已保存")

In [ ]:
# 展示单只股票的详细回测
def plot_single_stock_backtest(code, data_df):
    """绘制单只股票的回测详情"""
    
    stock_df = data_df[data_df['code'] == code].copy()
    stock_df = stock_df.sort_values('date')
    
    # 运行策略
    strategy = DualMAStrategy(short_window=5, long_window=20)
    result = strategy.backtest(stock_df)
    df = result['df']
    
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    
    # 1. 价格和均线
    ax = axes[0]
    ax.plot(df['date'], df['close'], label='Close', alpha=0.7)
    ax.plot(df['date'], df['ma_short'], label=f'MA{strategy.short_window}', alpha=0.7)
    ax.plot(df['date'], df['ma_long'], label=f'MA{strategy.long_window}', alpha=0.7)
    
    # 标记买卖点
    buy_signals = df[df['action'] == 'buy']
    sell_signals = df[df['action'] == 'sell']
    ax.scatter(buy_signals['date'], buy_signals['close'], marker='^', color='green', s=100, label='Buy', zorder=5)
    ax.scatter(sell_signals['date'], sell_signals['close'], marker='v', color='red', s=100, label='Sell', zorder=5)
    
    ax.set_ylabel('Price')
    ax.set_title(f'{code} Dual MA Strategy Backtest')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 2. 信号
    ax = axes[1]
    ax.fill_between(df['date'], 0, df['signal'], where=df['signal']==1, alpha=0.3, color='green', label='Long')
    ax.fill_between(df['date'], 0, df['signal'], where=df['signal']==-1, alpha=0.3, color='red', label='Short/Neutral')
    ax.set_ylabel('Signal')
    ax.set_title('Position Signal')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 3. 累计收益
    ax = axes[2]
    df['cum_return'] = (1 + df['close'].pct_change()).cumprod() - 1
    ax.plot(df['date'], df['cum_return']*100, label='Buy & Hold', alpha=0.7)
    ax.set_ylabel('Cumulative Return (%)')
    ax.set_xlabel('Date')
    ax.set_title(f'Cumulative Return: Strategy Return={result["total_return"]*100:.2f}%')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'./data/backtest_{code}.png', dpi=150)
    plt.show()
    
    return result

# 选择表现最好和最差的股票展示
best_code = test_results.loc[test_results['return'].idxmax(), 'code']
worst_code = test_results.loc[test_results['return'].idxmin(), 'code']

print(f"\n📈 展示表现最好的股票: {best_code}")
plot_single_stock_backtest(best_code, test_df)

print(f"\n📉 展示表现最差的股票: {worst_code}")
plot_single_stock_backtest(worst_code, test_df)

## 总结

In [ ]:
print("=" * 70)
print("📊 双均线策略量化流程总结")
print("=" * 70)

print("\n1️⃣ 数据加载")
print(f"   - 数据源: {DATA_PATH}")
print(f"   - 总记录数: {len(df):,}")

print("\n2️⃣ 股票筛选")
print(f"   - 筛选条件: 市值100-500亿，主板股票")
print(f"   - 选中股票数: {len(selected_codes)}")
print(f"   - 股票代码: {', '.join(selected_codes[:5])}...")

print("\n3️⃣ 数据集划分（时间序列，无穿越）")
print(f"   - 训练集: {train_df['date'].min().date()} ~ {train_df['date'].max().date()}")
print(f"   - 验证集: {val_df['date'].min().date()} ~ {val_df['date'].max().date()}")
print(f"   - 测试集: {test_df['date'].min().date()} ~ {test_df['date'].max().date()}")

print("\n4️⃣ 双均线策略回测结果")
print(f"   - 策略: MA5 上穿 MA20 买入，下穿卖出")
print(f"   - 训练集平均收益: {train_results['return'].mean()*100:.2f}%")
print(f"   - 验证集平均收益: {val_results['return'].mean()*100:.2f}%")
print(f"   - 测试集平均收益: {test_results['return'].mean()*100:.2f}%")

print("\n5️⃣ 输出文件")
print(f"   - 划分数据: train_data.csv, val_data.csv, test_data.csv")
print(f"   - 可视化: time_series_split.png, backtest_results_comparison.png")
print(f"   - 单股票回测: backtest_[code].png")

print("\n" + "=" * 70)